In [1]:
import torch
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
import sys
sys.path.insert(1,'C:\Zmyfiles\codes\research\pyDeepInsight_SRT\pyDeepInsight_SRT\examples\Scripts')


In [2]:
#import dataset
#SOURCE
expr_file1= r"C:\Zmyfiles\codes\research\pyDeepInsight_SRT\pyDeepInsight_SRT\examples\data\TvsN\TCGA_BRCA_NvT.csv"
expr1 = pd.read_csv(expr_file1, encoding='utf-8', engine='python')
expr1=expr1.transpose()
expr1.columns=expr1.iloc[0]
X1 = expr1.iloc[1:,0:-1].values
X_S=X1.astype(float)
X_S=X_S.T

#TARGET
expr_file2= r"C:\Zmyfiles\codes\research\pyDeepInsight_SRT\pyDeepInsight_SRT\examples\Data\TvsN\GSE45498_NvT.csv"
expr2 = pd.read_csv(expr_file2, encoding='utf-8', engine='python')
expr2=expr2.transpose()
expr2.columns=expr2.iloc[0]
X2 = expr2.iloc[1:,0:-1].values
X_T=X2.astype(float)
X_T=X_T.T

In [3]:
###Normalize the source and target dataset
# scaler = StandardScaler()
# X_S = scaler.fit_transform(X_S)
# X_T = scaler.fit_transform(X_T)
genes1 = expr1.iloc[:,:-1].columns.to_numpy()
genes2 = expr2.iloc[:,:-1].columns.to_numpy()

#extract common genes
# common_indices_X_S = []
# common_indices_X_T = []
# for i, row in enumerate(X_S):
#     for j, candidate in enumerate(X_T):
 
#         if np.array_equal(row, candidate):
#             common_indices_X_S.append(i)
#             common_indices_X_T.append(j)





In [4]:
#extract common genes
common_genes=[]
for i in genes1:
    for j in genes2:
        if i==j:
            common_genes.append(i)

print(common_genes)


condition1=np.isin(genes1,common_genes)
indexS=np.where(condition1)[0]

condition2=np.isin(genes2,common_genes)
indexT=np.where(condition2)[0]

['ABCB1', 'ABL1', 'AKT1', 'AKT2', 'APC', 'AR', 'AREG', 'ATM', 'BCL2', 'BCL2A1', 'BCL2L1', 'BCL3', 'BCL6', 'BCR', 'BIRC2', 'BIRC5', 'BLM', 'BMI1', 'BRAF', 'BRCA1', 'BRCA2', 'CASP10', 'CASP2', 'CAV1', 'CCNA2', 'CCND1', 'CCND2', 'CCND3', 'CCNE1', 'CD34', 'CD44', 'CDC25B', 'CDC25C', 'CDH1', 'CDH11', 'CDK2', 'CDK4', 'CDK6', 'CDKN1A', 'CDKN2A', 'CDKN2B', 'CDKN2C', 'CEBPA', 'CHEK1', 'CLTC', 'COL1A1', 'CSF1R', 'CSF3R', 'CSK', 'CTNNB1', 'CXCL9', 'CYP1A1', 'DAP3', 'DAPK1', 'DEK', 'DLC1', 'E2F1', 'E2F3', 'EGF', 'EGFR', 'EGR1', 'EPS8', 'ERBB2', 'ERBB3', 'ERBB4', 'ERCC2', 'ERCC4', 'ESR1', 'ETS1', 'ETS2', 'ETV1', 'ETV6', 'FANCG', 'FAS', 'FAT1', 'FGF1', 'FGF2', 'FGFR1', 'FGFR2', 'FGFR3', 'FGFR4', 'FGR', 'FLT1', 'FLT3', 'FOLR1', 'FOS', 'FOSL2', 'FRZB', 'FYN', 'GADD45A', 'GAPDH', 'GAS1', 'GATA1', 'GNAS', 'GRB7', 'GUSB', 'HCK', 'HDAC1', 'HIF1A', 'HMMR', 'HPRT1', 'HRAS', 'HSP90AB1', 'IFNGR1', 'IGF1', 'IGFBP2', 'IGFBP3', 'IGFBP6', 'IL1A', 'IL1B', 'IL4', 'IL6', 'IRF1', 'ITGB1', 'JUN', 'JUNB', 'KDR', 'KIT',

In [5]:
len(common_genes)

228

In [6]:
XsShared=X_S[indexS]
XsShared

array([[  7.58534064,  14.49738693,   1.15254052, ...,  19.26433612,
          1.76097646,   0.36639803],
       [ 49.18454883,  87.73684475,  70.05231361, ...,  76.42104212,
         77.93542976,  33.8281129 ],
       [ 65.55878631,  33.57000477,  45.95776103, ...,  49.25527953,
         52.05932146,  31.25589901],
       ...,
       [252.506226  , 248.3457916 , 171.0445767 , ..., 255.0027303 ,
        299.24642   , 286.8080727 ],
       [ 26.12959461,  69.48710156, 181.0546566 , ...,  68.93482732,
         63.77680642,  52.37160974],
       [ 33.80512651,  66.62545724,  57.5543313 , ...,  68.62313607,
         81.20369552,  52.96582363]])

In [7]:
# Example data
kpca = KernelPCA(n_components=2, kernel='rbf',gamma=.5)
Z_s = kpca.fit_transform(X_S)
Z_t = kpca.fit_transform(X_T)   # Target domain features
X_t = X_T.T                     # Original target domain data

Z_s = torch.tensor(Z_s, dtype=torch.float32)
Z_t = torch.tensor(Z_t, dtype=torch.float32, requires_grad=True)  # Make Z_t trainable
X_t = torch.tensor(X_t, dtype=torch.float32)

# Extract common points from Z1 and Z2
Zs_common = Z_s[indexS]
Zt_common = Z_t[indexT]

In [ ]:
Zt_common

In [8]:
# Define the RBF kernel
def rbf_kernel(x, y, gamma=1.0):
    pairwise_distances = torch.cdist(x, y) ** 2
    return torch.exp(-gamma * pairwise_distances)


In [9]:
# Define the MMD loss
def mmd_loss(Zs_common, Zt_common, gamma=1.0):
    # if(torch.isnan(Zs_common) or torch.isinf(Zs_common)):
    #     print("NaN Error in Source")
    #     exit()

    # if(torch.isnan(Zt_common) or torch.isinf(Zt_common)):
    #     print("NaN Error in Target")
    #     exit()

    K_ss = rbf_kernel(Zs_common, Zs_common, gamma)
    K_tt = rbf_kernel(Zt_common, Zt_common, gamma)
    K_st = rbf_kernel(Zs_common, Zt_common, gamma)
    
    mmd = K_ss.mean() + K_tt.mean() - 2 * K_st.mean()
    return mmd


In [10]:
# Define the regularization term
def regularization_term(Z_t, X_t, gamma=1.0):
    #print(Z_t.shape, X_t.shape)
    K_t = rbf_kernel(X_t.T, X_t.T, gamma)
    #print(K_t.shape)
    A = Z_t.T @ K_t @ Z_t
    trace_A = torch.trace(A)
    return trace_A

In [11]:
# Hyperparameters
lambda_reg = 0.1  # Regularization strength
gamma = 1.0       # Kernel parameter

# Optimizer
optimizer = optim.SGD([Z_t], lr=0.01)


In [12]:
# Training loop
for epoch in range(100):
    Z_t = Z_t.detach().requires_grad_(True)
    optimizer.zero_grad()
    
    # Compute MMD loss
    Zt_common = Z_t[indexT]
    mmd = mmd_loss(Zs_common, Zt_common, gamma)
    
    # Compute regularization term
    reg = regularization_term(Z_t, X_t, gamma)
    
    # Total loss
    loss = mmd + lambda_reg * reg
    
    # Backpropagate and update Z_t
    loss.backward(retain_graph=True)
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, MMD: {mmd.item():.4f}, Reg: {reg.item():.4f}")

# Final Z_t after optimization
print("Optimized Z_t:", Z_t)

Epoch 0, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 10, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 20, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 30, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 40, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 50, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 60, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 70, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 80, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Epoch 90, Loss: 0.1618, MMD: 0.0009, Reg: 1.6089
Optimized Z_t: tensor([[-4.7581e-02, -3.0694e-03],
        [-1.2648e-02,  3.3323e-02],
        [ 9.3135e-02,  3.2439e-03],
        [-7.8586e-02, -7.1732e-02],
        [-2.4842e-02,  6.0469e-02],
        [-1.5142e-01,  4.5118e-02],
        [-4.8203e-02,  2.8969e-03],
        [ 6.4289e-03,  6.7875e-02],
        [ 9.4374e-04,  1.6583e-02],
        [ 4.6686e-02,  6.0377e-02],
        [-1.9267e-02,  3.6711e-02],
        [ 7.0085e-02, -7.8137e-02],
        [ 1.2153e-01, -2.3686e-02],
        [ 2.0008e-01, -7.503